# MAI-Image-2.5 Family — Combined Developer Demo

A self-contained walkthrough covering all three deployed MAI-Image-2.5 models on Microsoft Foundry: `mai-image-2.5`, `mai-image-2.5-flash`, and `mai-image-2.5-pro`.

1. Install dependencies
2. Configure credentials (all three deployments)
3. Define shared helpers
4. **MAI-Image-2.5-Flash** — clothing resale listing backgrounds, AI Engineer's desk mood boards
5. **MAI-Image-2.5** — image editing, before/after comparisons
6. **MAI-Image-2.5-Pro** — text-to-image, character-consistency evaluation

Run the cells in order from top to bottom — later sections reuse the credentials and helpers defined at the top.

## Step 1 — Install dependencies

In [ ]:
%pip install -q requests python-dotenv pillow ipywidgets

## Step 2 — Configure credentials

The cell loads `.env` from this folder first. Add these values:

```text
MICROSOFT_FOUNDRY_ENDPOINT=https://<resource-name>.services.ai.azure.com
MICROSOFT_FOUNDRY_API_KEY=<your-api-key>
AZURE_MAI_IMAGE_25_DEPLOYMENT=mai-image-2.5
AZURE_MAI_IMAGE_25_FLASH_DEPLOYMENT=mai-image-2.5-flash
AZURE_MAI_IMAGE_25_PRO_DEPLOYMENT=mai-image-2.5-pro
```

Alternatively, uncomment the notebook-only overrides in the next cell.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# Notebook-only alternative to .env:
# os.environ['MICROSOFT_FOUNDRY_ENDPOINT'] = 'https://<resource-name>.services.ai.azure.com'
# os.environ['MICROSOFT_FOUNDRY_API_KEY'] = '<your-api-key>'
# os.environ['AZURE_MAI_IMAGE_25_DEPLOYMENT'] = 'mai-image-2.5'
# os.environ['AZURE_MAI_IMAGE_25_FLASH_DEPLOYMENT'] = 'mai-image-2.5-flash'
# os.environ['AZURE_MAI_IMAGE_25_PRO_DEPLOYMENT'] = 'mai-image-2.5-pro'

ENDPOINT = os.getenv('MICROSOFT_FOUNDRY_ENDPOINT', '').rstrip('/')
MODEL_API_KEY = os.getenv('MICROSOFT_FOUNDRY_API_KEY', '')
DEPLOYMENT_STANDARD = os.getenv('AZURE_MAI_IMAGE_25_DEPLOYMENT', '')
DEPLOYMENT_FLASH = os.getenv('AZURE_MAI_IMAGE_25_FLASH_DEPLOYMENT', '')
DEPLOYMENT_PRO = os.getenv('AZURE_MAI_IMAGE_25_PRO_DEPLOYMENT', '')
missing = [name for name, value in {
    'MICROSOFT_FOUNDRY_ENDPOINT': ENDPOINT,
    'MICROSOFT_FOUNDRY_API_KEY': MODEL_API_KEY,
    'AZURE_MAI_IMAGE_25_DEPLOYMENT': DEPLOYMENT_STANDARD,
    'AZURE_MAI_IMAGE_25_FLASH_DEPLOYMENT': DEPLOYMENT_FLASH,
    'AZURE_MAI_IMAGE_25_PRO_DEPLOYMENT': DEPLOYMENT_PRO,
}.items() if not value]
if missing:
    raise EnvironmentError(f'Missing credential(s): {", ".join(missing)}')

print(f'Endpoint: {ENDPOINT}')
print(f'Standard deployment: {DEPLOYMENT_STANDARD}')
print(f'Flash deployment: {DEPLOYMENT_FLASH}')
print(f'Pro deployment: {DEPLOYMENT_PRO}')
print(f'API key: ***{MODEL_API_KEY[-4:]}')

## Step 3 — Shared helpers

These helpers send authenticated requests, surface service error details, avoid overwriting outputs, and decode the returned base64 images. Every section below reuses them.

In [ ]:
import base64
import binascii
import time
from pathlib import Path

import requests

REQUEST_TIMEOUT = 300

def versioned_path(path: str) -> str:
    candidate = Path(path)
    if not candidate.exists():
        return str(candidate)
    for version in range(2, 10_000):
        next_path = candidate.with_stem(f'{candidate.stem}_v{version}')
        if not next_path.exists():
            return str(next_path)
    raise RuntimeError('Could not find an available output path.')

def post_image_api(route: str, *, json=None, data=None, files=None) -> dict:
    started = time.perf_counter()
    response = requests.post(f'{ENDPOINT}/mai/v1/images/{route}', headers={'api-key': MODEL_API_KEY}, json=json, data=data, files=files, timeout=REQUEST_TIMEOUT)
    if not response.ok:
        raise requests.HTTPError(f'{response.status_code} {response.reason}: {response.text}', response=response)
    print(f'API latency: {time.perf_counter() - started:.2f}s')
    return response.json()

def save_images(result: dict, output_path: str) -> list[str]:
    images = [item['b64_json'] for item in result.get('data', []) if item.get('b64_json')]
    if not images:
        raise ValueError(f'Unexpected response format: {result}')
    output = Path(output_path)
    output.parent.mkdir(parents=True, exist_ok=True)
    paths = []
    for index, encoded in enumerate(images, start=1):
        path = output if len(images) == 1 else output.with_stem(f'{output.stem}_{index}')
        try:
            path.write_bytes(base64.b64decode(encoded))
        except (binascii.Error, ValueError) as exc:
            raise ValueError(f'Could not decode image {index}.') from exc
        paths.append(str(path.resolve()))
        print(f'Saved: {paths[-1]}')
    return paths

# MAI-Image-2.5-Flash

Fast concurrent generation and editing using the `mai-image-2.5-flash` deployment.

## Clothing resale listing backgrounds

Create six marketplace-ready background variations for a secondhand clothing photo. The workflow issues 6 API requests concurrently with `ThreadPoolExecutor` and preserves the item while replacing only the background.

In [ ]:
DEPLOYMENT_NAME = DEPLOYMENT_FLASH
print(f'Model: {DEPLOYMENT_NAME}')

import concurrent.futures
import mimetypes
import ipywidgets as widgets
from IPython.display import display

MAX_WORKERS = 4  # increase after confirming your quota allows it

RESALE_LISTING_BACKGROUNDS = [
    ('paris_apartment', 'a bright Parisian apartment with pale walls, subtle molding, and warm oak flooring'),
    ('fashion_boutique', 'an elegant independent fashion boutique with neutral decor and soft window light'),
    ('minimal_studio', 'a clean warm-white photography studio with a seamless backdrop and soft diffused lighting'),
    ('luxury_wardrobe', 'a refined walk-in wardrobe with muted finishes and an uncluttered editorial look'),
    ('hotel_suite', 'a sophisticated boutique hotel suite with tasteful neutral furnishings and natural daylight'),
    ('stone_townhouse', 'a chic European townhouse interior with pale stone details and understated styling'),
]

def create_resale_listing_background_variations(image_path: str, output_dir: str = 'output') -> list[str]:
    source = Path(image_path)
    if not source.is_file():
        raise FileNotFoundError(f'Clothing image not found: {source}')
    mime_type = mimetypes.guess_type(source.name)[0]
    if mime_type not in {'image/jpeg', 'image/png'}:
        raise ValueError('The clothing image must be a JPEG or PNG image.')

    def create_variation(item: tuple[int, tuple[str, str]]) -> list[str]:
        index, (name, background) = item
        prompt = ('Edit only the background of this secondhand clothing photo. Place the clothing item in ' + background + '. ' 'Keep the item and every foreground detail unchanged: shape, cut, proportions, color, pattern, fabric texture, seams, labels, embellishments, condition, and fit. ' 'Do not redesign, retouch, reshape, recolor, or add anything to the item. Match the original perspective and use physically realistic lighting and shadows. ' 'Create an authentic professional product photograph suitable for an online clothing resale listing. No text, logos, borders, or watermarks.')
        output_path = versioned_path(str(Path(output_dir) / f'{source.stem}_resale_listing_backgrounds' / f'{index:02d}_{name}.png'))
        print(f'Creating variation {index}/{len(RESALE_LISTING_BACKGROUNDS)}: {name}')
        with source.open('rb') as image_file:
            result = post_image_api('edits', data={'model': DEPLOYMENT_NAME, 'prompt': prompt}, files={'image': (source.name, image_file, mime_type)})
        return save_images(result, output_path)

    indexed_backgrounds = list(enumerate(RESALE_LISTING_BACKGROUNDS, start=1))
    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        variation_paths = executor.map(create_variation, indexed_backgrounds)
    return [path for paths in variation_paths for path in paths]

CLOTHING_IMAGE = 'images/blue-dress.jpg'  # Replace with your clothing image.
resale_listing_paths = create_resale_listing_background_variations(CLOTHING_IMAGE)
display(widgets.GridBox([widgets.VBox([widgets.Label(Path(path).stem), widgets.Image(value=Path(path).read_bytes(), format='png', layout=widgets.Layout(width='100%'))]) for path in resale_listing_paths], layout=widgets.Layout(grid_template_columns='repeat(2, minmax(0, 1fr))', grid_gap='16px')))

## AI Engineer's desk mood boards

Create six editorial mood boards for **The AI Engineer's desk**. Art directions are generated from customer-supplied keywords rather than a fixed list, so each run can produce a differentiated set. This workflow also issues 6 API requests concurrently.

In [ ]:
import random

# Style modifiers combined with customer keywords to build each art direction.
MOOD_BOARD_STYLE_MODIFIERS = [
    ('precision', 'white ceramic, brushed aluminum, cool daylight, cyan accents'),
    ('warm', 'walnut desk, soft amber light, plants, coral and teal accents'),
    ('midnight', 'charcoal surfaces, focused monitor glow, electric blue accents'),
    ('calm', 'pale ash wood, matte white tools, diffused morning light, sage accents'),
    ('analog', 'notebooks, technical sketches, mechanical keyboard, moss and rust tones'),
    ('luxury', 'smoked glass, black metal, sculptural lighting, silver accents'),
    ('industrial', 'raw concrete, exposed steel, harsh overhead light, rust accents'),
    ('coastal', 'bleached driftwood, linen textures, soft diffused light, seafoam accents'),
]

def generate_desk_directions(keywords: list[str], count: int = 6, seed: int | None = None) -> list[tuple[str, str]]:
    """Pair customer keywords with shuffled style modifiers to build `count` distinct art directions."""
    if not keywords:
        raise ValueError('Provide at least one customer keyword.')
    rng = random.Random(seed)
    modifiers = MOOD_BOARD_STYLE_MODIFIERS.copy()
    rng.shuffle(modifiers)
    keyword_cycle = keywords * (count // len(keywords) + 1)
    rng.shuffle(keyword_cycle)
    directions = []
    for index in range(count):
        keyword = keyword_cycle[index]
        style_name, style_details = modifiers[index % len(modifiers)]
        name = f'{style_name}_{keyword}'.replace(' ', '_').lower()
        direction = f'{keyword}-inspired workspace, {style_details}'
        directions.append((name, direction))
    return directions

CUSTOMER_KEYWORDS = ['Feminine', 'plants', 'comfortable']  # replace with customer-provided keywords
AI_ENGINEER_DESK_DIRECTIONS = generate_desk_directions(CUSTOMER_KEYWORDS, count=6)  # pass seed=<int> to reproduce a specific set

In [ ]:
def generate_ai_engineer_desk_mood_boards(output_dir: str = 'output/ai_engineers_desk_mood_boards') -> list[str]:
    def generate_mood_board(item: tuple[int, tuple[str, str]]) -> list[str]:
        index, (name, direction) = item
        prompt = (
            "Create a sophisticated visual mood board for the concept 'The AI Engineer's desk'. "
            f'Art direction: {direction}. Compose a cohesive editorial collage with one hero workspace image and smaller supporting frames showing tactile materials, desk objects, computing hardware, abstract neural-network diagrams, lighting details, and a restrained row of color swatches. '
            'Professional design presentation, photorealistic product photography, precise grid, generous spacing, coherent lighting, premium art-direction quality, square composition. No people, brand logos, watermarks, captions, labels, or legible text.'
        )
        print(f'Creating mood board {index}/{len(AI_ENGINEER_DESK_DIRECTIONS)}: {name}')
        result = post_image_api('generations', json={'model': DEPLOYMENT_NAME, 'prompt': prompt, 'width': 1024, 'height': 1024})
        return save_images(result, versioned_path(str(Path(output_dir) / f'{index:02d}_{name}.png')))

    indexed_directions = list(enumerate(AI_ENGINEER_DESK_DIRECTIONS, start=1))
    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        mood_board_paths = executor.map(generate_mood_board, indexed_directions)
    return [path for paths in mood_board_paths for path in paths]

mood_board_paths = generate_ai_engineer_desk_mood_boards()
display(widgets.GridBox([widgets.VBox([widgets.Label(Path(path).stem), widgets.Image(value=Path(path).read_bytes(), format='png', layout=widgets.Layout(width='100%'))]) for path in mood_board_paths], layout=widgets.Layout(grid_template_columns='repeat(2, minmax(0, 1fr))', grid_gap='16px')))

# MAI-Image-2.5

Image editing using the base `mai-image-2.5` deployment, plus a static before/after showcase.

## Image editing

Upload one reference photo to `/mai/v1/images/edits` and describe the edit.

In [ ]:
DEPLOYMENT_NAME = DEPLOYMENT_STANDARD
print(f'Model: {DEPLOYMENT_NAME}')

import mimetypes
from IPython.display import Image, display

INPUT_IMAGE = 'images/car2.png'  # Replace with your JPEG or PNG file.
EDIT_PROMPT = 'Remove the finger blur mark while preserving every other detail of this photograph.'
input_path = Path(INPUT_IMAGE)
if not input_path.is_file():
    raise FileNotFoundError(f'Input image not found: {input_path}')
mime_type = mimetypes.guess_type(input_path.name)[0] or 'image/png'
if mime_type not in {'image/jpeg', 'image/png'}:
    raise ValueError('Use a JPEG or PNG input image.')

edited_output_path = versioned_path(f'output/{input_path.stem}_edited.png')
with input_path.open('rb') as image_file:
    result = post_image_api('edits', data={'model': DEPLOYMENT_NAME, 'prompt': EDIT_PROMPT}, files={'image': (input_path.name, image_file, mime_type)})
edited_paths = save_images(result, edited_output_path)
display(Image(filename=edited_paths[0]))

## Compare original vs. edited image

In [ ]:
import ipywidgets as widgets

# Fixed pixel width so both previews render at the same on-screen size regardless of source resolution.
PREVIEW_WIDTH = '480px'

def preview_image(path: str, label: str, image_format: str) -> widgets.VBox:
    return widgets.VBox([
        widgets.Label(label),
        widgets.Image(value=Path(path).read_bytes(), format=image_format, layout=widgets.Layout(width=PREVIEW_WIDTH, height='auto')),
    ])

original_format = 'jpeg' if mime_type == 'image/jpeg' else 'png'
display(widgets.HBox([
    preview_image(str(input_path), 'Original', original_format),
    preview_image(edited_paths[0], 'Edited', 'png'),
], layout=widgets.Layout(justify_content='space-around')))

# MAI-Image-2.5-Pro

Text-to-image generation, followed by a full character-consistency evaluation — the focus of this section — using the `mai-image-2.5-pro` deployment.

## Text-to-image generation

In [ ]:
DEPLOYMENT_NAME = DEPLOYMENT_PRO
print(f'Model: {DEPLOYMENT_NAME}')

PROMPT = 'A premium editorial product photograph of a sculptural glass lamp on a travertine pedestal, warm afternoon sunlight, soft shadows, photorealistic, no text or watermark.'
WIDTH, HEIGHT = 1024, 1024
output_path = versioned_path('output/pro_text_to_image.png')

result = post_image_api('generations', json={'model': DEPLOYMENT_NAME, 'prompt': PROMPT, 'width': WIDTH, 'height': HEIGHT})
generated_paths = save_images(result, output_path)
display(Image(filename=generated_paths[0]))

## Character-consistency evaluation (focus)

Uses `images/bit1.png` as the exact character reference for **Bit**. First, three new scenes are generated preserving Bit's design. Then a full 27-case test matrix probes consistency across repeatability, camera angle, lighting, occlusion/scale, wardrobe preservation, lens framing, object interaction, prompt pressure, and pose/action — followed by a contact sheet and a manual scoring template.

> **WARNING**
>
> Before you run the Pro model understand the costs.
>
> Using the published MAI-Image-2.5-Pro rates:
>* Text input: $5 per 1M tokens
>* Image output: $106 per 1M tokens
> 
> Estimate a 1024 x 1024 image input in this sample - ~$0.45 per request
>
> Running this cell makes 3 MAI-Image-2.5-Pro API calls and may cost approximately **$1.40 per run**.

### Reference scenes

In [ ]:
BIT_REFERENCE_IMAGE = Path('images/bit1.png')
if not BIT_REFERENCE_IMAGE.is_file():
    raise FileNotFoundError(f'Bit reference image not found: {BIT_REFERENCE_IMAGE}')

bit_mime_type = mimetypes.guess_type(BIT_REFERENCE_IMAGE.name)[0] or 'image/png'
if bit_mime_type not in {'image/jpeg', 'image/png'}:
    raise ValueError('Use a JPEG or PNG reference image.')

BIT_IDENTITY = (
    'Use the provided image as the exact character reference for Bit. Preserve Bit as the same gray plush raccoon character: '
    'identical face, dark eye mask, large round white eyes, small ears with pink interiors, gray fur, body proportions, '
    'turquoise Azure-logo shirt, friendly expression, and soft plush-toy texture. Do not redesign, recolor, or replace the character. '
)

BIT_SCENES = {
    'technical_conference_stage': (
        'Create a photorealistic event photograph of Bit presenting on a modern technical conference stage, standing beside a large screen '
        'showing abstract cloud architecture diagrams with no legible text. Include professional stage lighting, a lectern, and an engaged audience '
        'seen from behind. Keep Bit clearly visible as the featured speaker.'
    ),
    'conference_booth': (
        'Create a photorealistic event photograph of Bit welcoming attendees at a polished technology conference booth. Include demo monitors, '
        'a clean counter, subtle cloud-computing visuals with no legible text, and attendees interacting nearby. Keep Bit clearly visible in front '
        'of the booth as the host.'
    ),
    'hackathon': (
        'Create a photorealistic candid photograph of Bit participating in an energetic hackathon, seated with a small team around laptops, cables, '
        'sticky notes, and prototype electronics. Show a collaborative late-night workspace with warm practical lighting. Keep Bit clearly visible '
        'as an active member of the team.'
    ),
}

bit_scene_paths = {}
for scene_name, scene_prompt in BIT_SCENES.items():
    print(f'Creating Bit scene: {scene_name}')
    with BIT_REFERENCE_IMAGE.open('rb') as image_file:
        result = post_image_api(
            'edits',
            data={'model': DEPLOYMENT_NAME, 'prompt': BIT_IDENTITY + scene_prompt},
            files={'image': (BIT_REFERENCE_IMAGE.name, image_file, bit_mime_type)},
        )
    bit_scene_paths[scene_name] = save_images(
        result,
        versioned_path(f'output/bit_{scene_name}.png'),
    )[0]

for scene_name, image_path in bit_scene_paths.items():
    print(scene_name.replace('_', ' ').title())
    display(Image(filename=image_path, width=420))

### Consistency test matrix definition

In [ ]:
# Character-consistency test matrix: define cases without making API requests.

RUN_MATRIX = True

MATRIX_OUTPUT_ROOT = Path('output/bit_consistency_matrix')

BIT_IDENTITY_SHORT = (
    'Keep Bit recognizably the same character as the reference image, including the face, gray fur, proportions, turquoise shirt, and Azure logo. '
)

BIT_CONSISTENCY_CASES = [
    {
        'group': 'repeatability',
        'case': f'neutral_{index:02d}',
        'prompt': (
            'Create a neutral full-body studio photograph of Bit standing upright and facing the camera, centered against a plain light-gray background, '
            'with soft even lighting and no props, text, or other characters.'
        ),
    }
    for index in range(1, 4)
] + [
    {
        'group': 'camera_angle',
        'case': 'front',
        'prompt': 'Create a neutral full-body studio photograph of Bit viewed directly from the front, with soft even lighting and a plain light-gray background.',
    },
    {
        'group': 'camera_angle',
        'case': 'side_profile',
        'prompt': 'Create a neutral full-body studio photograph of Bit viewed in a clear left-side profile, with soft even lighting and a plain light-gray background.',
    },
    {
        'group': 'camera_angle',
        'case': 'rear_three_quarter',
        'prompt': 'Create a neutral full-body studio photograph of Bit viewed from a rear three-quarter angle while Bit turns their face toward the camera, with soft even lighting and a plain light-gray background.',
    },
    {
        'group': 'lighting',
        'case': 'daylight',
        'prompt': 'Create a full-body photograph of Bit facing the camera beside a large window in natural daylight, with a simple neutral room and no other characters.',
    },
    {
        'group': 'lighting',
        'case': 'conference_stage',
        'prompt': 'Create a full-body photograph of Bit facing the camera under cool technical-conference stage lighting, with a minimal dark stage and no other characters.',
    },
    {
        'group': 'lighting',
        'case': 'warm_low_light',
        'prompt': 'Create a full-body photograph of Bit facing the camera in warm low indoor light, preserving clearly visible facial features, colors, and clothing.',
    },
    {
        'group': 'occlusion_scale',
        'case': 'close_up',
        'prompt': 'Create a close-up portrait of Bit facing the camera, framed from the chest upward against a plain neutral background.',
    },
    {
        'group': 'occlusion_scale',
        'case': 'behind_desk',
        'prompt': 'Create a photograph of Bit facing the camera while standing behind a simple desk that partially hides the lower body, in a neutral office with no other characters.',
    },
    {
        'group': 'occlusion_scale',
        'case': 'distant_crowd',
        'prompt': 'Create a wide conference photograph with Bit facing the camera and fully visible at a distance among a small crowd, while keeping Bit recognizable and unobstructed.',
    },
    {
        'group': 'wardrobe_preservation',
        'case': 'conference_lanyard',
        'prompt': 'Create a full-body conference portrait of Bit wearing a simple badge lanyard over the original turquoise Azure-logo shirt; keep the shirt, logo, colors, and character design unchanged and clearly visible.',
    },
    {
        'group': 'wardrobe_preservation',
        'case': 'open_jacket',
        'prompt': 'Create a full-body portrait of Bit wearing an open lightweight jacket over the original turquoise Azure-logo shirt; preserve the original shirt and logo without redesigning or recoloring them.',
    },
    {
        'group': 'wardrobe_preservation',
        'case': 'backpack',
        'prompt': 'Create a full-body portrait of Bit wearing a small backpack while the original turquoise Azure-logo shirt remains unobstructed, unchanged, and clearly visible.',
    },
    {
        'group': 'lens_framing',
        'case': 'wide_angle',
        'prompt': 'Create a full-body environmental portrait of Bit using a 24mm wide-angle lens from a moderate distance, centered in a simple conference hallway without distorting the character.',
    },
    {
        'group': 'lens_framing',
        'case': 'normal_portrait',
        'prompt': 'Create a three-quarter portrait of Bit using a natural 50mm lens perspective, centered against a simple neutral conference background.',
    },
    {
        'group': 'lens_framing',
        'case': 'telephoto_close_up',
        'prompt': 'Create a chest-up portrait of Bit using an 85mm telephoto portrait lens with gentle background blur and undistorted facial proportions.',
    },
    {
        'group': 'object_interaction',
        'case': 'holding_microphone',
        'prompt': 'Create a full-body photograph of Bit naturally holding a handheld microphone while facing the camera on a simple conference stage.',
    },
    {
        'group': 'object_interaction',
        'case': 'using_laptop',
        'prompt': 'Create a three-quarter photograph of Bit actively typing on an open laptop at a simple desk while looking toward the camera.',
    },
    {
        'group': 'object_interaction',
        'case': 'carrying_mug',
        'prompt': 'Create a full-body photograph of Bit naturally carrying a plain ceramic mug in one paw in a neutral office setting.',
    },
    {
        'group': 'prompt_pressure',
        'case': 'full_identity',
        'identity_prompt': BIT_IDENTITY,
        'prompt': 'Create a full-body photograph of Bit presenting beside a laptop in a modern conference room with soft daylight.',
    },
    {
        'group': 'prompt_pressure',
        'case': 'short_identity',
        'identity_prompt': BIT_IDENTITY_SHORT,
        'prompt': 'Create a full-body photograph of Bit presenting beside a laptop in a modern conference room with soft daylight.',
    },
    {
        'group': 'prompt_pressure',
        'case': 'reference_only',
        'identity_prompt': '',
        'prompt': 'Create a full-body photograph of Bit presenting beside a laptop in a modern conference room with soft daylight.',
    },
    {
        'group': 'pose_action',
        'case': 'standing_neutral',
        'prompt': 'Create a neutral full-body studio photograph of Bit standing upright with both arms relaxed, facing the camera against a plain light-gray background.',
    },
    {
        'group': 'pose_action',
        'case': 'waving',
        'prompt': 'Create a full-body studio photograph of Bit facing the camera and waving with one paw, with natural anatomy and a plain light-gray background.',
    },
    {
        'group': 'pose_action',
        'case': 'walking',
        'prompt': 'Create a full-body photograph of Bit walking naturally toward the camera in a simple conference hallway, preserving body proportions and recognizable features.',
    },
]

EXPECTED_MATRIX_CASES = 27
assert len(BIT_CONSISTENCY_CASES) == EXPECTED_MATRIX_CASES
assert len({f"{item['group']}_{item['case']}" for item in BIT_CONSISTENCY_CASES}) == EXPECTED_MATRIX_CASES
print(f'Matrix ready: {len(BIT_CONSISTENCY_CASES)} cases')
print(f'API requests enabled: {RUN_MATRIX}')

### Run matrix (rate-limited concurrent execution)

> **WARNING**
>
> Running this cell makes 27 MAI-Image-2.5-Pro API calls and may cost approximately **$12.30 per run**.

In [ ]:
# Run matrix requests concurrently while pacing request starts to the deployment's RPM quota.

from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

# Match these values to the RPM tier shown for the deployment in Microsoft Foundry.
MAX_CONCURRENT_REQUESTS = 2
REQUESTS_PER_MINUTE = 2

if MAX_CONCURRENT_REQUESTS < 1:
    raise ValueError('MAX_CONCURRENT_REQUESTS must be at least 1.')
if REQUESTS_PER_MINUTE < 1:
    raise ValueError('REQUESTS_PER_MINUTE must be at least 1.')

matrix_results = []
matrix_run_dir = None

if not RUN_MATRIX:
    print(f'Dry run only. Set RUN_MATRIX = True in the previous cell and rerun both cells to generate {len(BIT_CONSISTENCY_CASES)} images.')
else:
    if not BIT_REFERENCE_IMAGE.is_file():
        raise FileNotFoundError(f'Bit reference image not found: {BIT_REFERENCE_IMAGE}')

    run_stamp = time.strftime('%Y%m%d_%H%M%S')
    matrix_run_dir = Path(versioned_path(str(MATRIX_OUTPUT_ROOT / f'run_{run_stamp}')))
    matrix_run_dir.mkdir(parents=True)
    print(f'Writing matrix run to: {matrix_run_dir.resolve()}')
    print(f'Concurrency: {MAX_CONCURRENT_REQUESTS}; request rate: {REQUESTS_PER_MINUTE} RPM')

    request_interval = 60.0 / REQUESTS_PER_MINUTE
    next_request_at = [time.monotonic()]
    request_rate_lock = threading.Lock()

    def wait_for_request_slot():
        with request_rate_lock:
            wait_seconds = next_request_at[0] - time.monotonic()
            if wait_seconds > 0:
                time.sleep(wait_seconds)
            next_request_at[0] = time.monotonic() + request_interval

    def generate_matrix_case(index, item):
        case_id = f"{item['group']}_{item['case']}"
        output_file = matrix_run_dir / f'{case_id}.png'
        identity_prompt = item.get('identity_prompt', BIT_IDENTITY)
        effective_prompt = identity_prompt + item['prompt']

        try:
            wait_for_request_slot()
            with BIT_REFERENCE_IMAGE.open('rb') as image_file:
                result = post_image_api(
                    'edits',
                    data={'model': DEPLOYMENT_NAME, 'prompt': effective_prompt},
                    files={'image': (BIT_REFERENCE_IMAGE.name, image_file, bit_mime_type)},
                )
            generated_path = save_images(result, str(output_file))[0]
            status = 'generated'
            error = ''
        except Exception as exc:
            generated_path = ''
            status = 'failed'
            error = str(exc)

        return index, {
            **item,
            'identity_prompt': identity_prompt,
            'effective_prompt': effective_prompt,
            'status': status,
            'path': generated_path,
            'error': error,
        }

    ordered_results = [None] * len(BIT_CONSISTENCY_CASES)
    worker_count = min(MAX_CONCURRENT_REQUESTS, len(BIT_CONSISTENCY_CASES))
    with ThreadPoolExecutor(max_workers=worker_count) as executor:
        futures = [
            executor.submit(generate_matrix_case, index, item)
            for index, item in enumerate(BIT_CONSISTENCY_CASES)
        ]
        for completed_count, future in enumerate(as_completed(futures), start=1):
            index, case_result = future.result()
            ordered_results[index] = case_result
            case_id = f"{case_result['group']}_{case_result['case']}"
            print(f"[{completed_count:02d}/{len(futures)}] {case_id}: {case_result['status']}")
            if case_result['error']:
                print(f"  {case_result['error']}")

    matrix_results = ordered_results
    completed = sum(item['status'] == 'generated' for item in matrix_results)
    failed = sum(item['status'] == 'failed' for item in matrix_results)
    print(f'Generated {completed}/{len(BIT_CONSISTENCY_CASES)} images; {failed} failed.')

### Contact sheet & scorecard

In [ ]:
# Build a labeled contact sheet and a 0-2 manual scoring template after generation.

import csv
import math
import textwrap

from PIL import Image as PILImage, ImageDraw, ImageOps

SCORE_FIELDS = [
    'face',
    'eyes',
    'ears',
    'fur',
    'proportions',
    'shirt',
    'logo',
    'overall_identity',
    'scene_compliance',
    'notes',
]

if not matrix_results:
    print('No matrix results yet. Enable and run the previous two cells first.')
else:
    if matrix_run_dir is None:
        raise RuntimeError('Matrix results exist but the run directory is unavailable.')

    scorecard_path = matrix_run_dir / 'scorecard.csv'
    fieldnames = [
        'group',
        'case',
        'status',
        'path',
        'error',
        'identity_prompt',
        'prompt',
        'effective_prompt',
        *SCORE_FIELDS,
    ]
    with scorecard_path.open('w', newline='', encoding='utf-8') as scorecard_file:
        writer = csv.DictWriter(scorecard_file, fieldnames=fieldnames)
        writer.writeheader()
        for item in matrix_results:
            writer.writerow({**item, **{field: '' for field in SCORE_FIELDS}})

    generated_items = [item for item in matrix_results if item['status'] == 'generated']
    contact_items = [('Reference', str(BIT_REFERENCE_IMAGE)), *[
        (f"{item['group']} / {item['case']}", item['path'])
        for item in generated_items
    ]]

    tile_width, image_height, label_height = 320, 320, 60
    columns = 4
    rows = math.ceil(len(contact_items) / columns)
    contact_sheet = PILImage.new('RGB', (columns * tile_width, rows * (image_height + label_height)), 'white')
    draw = ImageDraw.Draw(contact_sheet)

    for index, (label, path) in enumerate(contact_items):
        column = index % columns
        row = index // columns
        tile_x = column * tile_width
        tile_y = row * (image_height + label_height)
        with PILImage.open(path) as source_image:
            prepared = ImageOps.exif_transpose(source_image).convert('RGB')
            prepared = ImageOps.contain(
                prepared,
                (tile_width - 16, image_height - 16),
                method=PILImage.Resampling.LANCZOS,
            )
        image_x = tile_x + (tile_width - prepared.width) // 2
        image_y = tile_y + (image_height - prepared.height) // 2
        contact_sheet.paste(prepared, (image_x, image_y))
        wrapped_label = '\n'.join(textwrap.wrap(label.replace('_', ' ').title(), width=34))
        draw.multiline_text((tile_x + 8, tile_y + image_height + 8), wrapped_label, fill='black', spacing=3)

    contact_sheet_path = matrix_run_dir / 'contact_sheet.jpg'
    contact_sheet.save(contact_sheet_path, quality=92)
    print(f'Scorecard: {scorecard_path.resolve()}')
    print(f'Contact sheet: {contact_sheet_path.resolve()}')
    display(Image(filename=str(contact_sheet_path)))